# Part I: Language model comparison

This section consists of 4 models:  
  
1. Interpolated n-gram  
2. Vanilla RNN  
3. LSTM  
4. Causal Transformer  
      
**Data:** *Great Expectations* by Charles Dickens (1861), Project Gutenberg eBook #1400, https://www.gutenberg.org/ebooks/1400. All are trained on the same corpus and evaluated on the same test set.

**How to run:** run `python src/prepare_data.py` from the project root once to create `data/processed/`, then run all cells in order (about 50 minutes on a CPU). All metrics, samples and figures are written to `results/part1/`.

**Shared evaluation:** every model predicts each test token from the preceding test tokens only. Perplexity = exp(mean cross-entropy), with padding excluded. Generation uses the same 4 prompts, 30 tokens, temperature 0.8 and seed 4213 + prompt index for every model.

Reusable code is in `src/`: `data`, `ngram`, `models`, `train`, `evaluate`, `experiment`.

## 0. Setup

Import libraries and check hardware details.

In [1]:
import sys
from pathlib import Path
import json
import statistics

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.data import load_corpus, tokenize, detokenize, unk_rate
from src.ngram import InterpolatedNgram
from src.models import RecurrentLM, TransformerLM
from src.evaluate import (
    SEED, PROMPTS, RESULTS_DIR,
    Timer, perplexity, count_parameters, generate_samples,
    save_results, hardware_summary, set_seed, neural_mean_nll
)
from src.experiment import run_neural_model, plot_history

set_seed(SEED)
hardware_summary()

{'platform': 'Windows-11-10.0.26200-SP0',
 'processor': 'Intel64 Family 6 Model 170 Stepping 4, GenuineIntel',
 'python': '3.13.9',
 'torch': '2.14.1+cpu',
 'torch_threads': 16,
 'device': 'cpu'}

## 1. Data

Load the vocabulary and the three encoded splits (train, validation, and test sets) from the corpus.

In [2]:
vocab, splits = load_corpus()
print(f"Vocabulary size: {len(vocab)}")

for name, ids in splits.items():
    print(f"{name} set: {len(ids)} tokens, {unk_rate(ids, vocab):.2%} unknown")

Vocabulary size: 4103
train set: 199118 tokens, 3.91% unknown
valid set: 12653 tokens, 5.84% unknown
test set: 9889 tokens, 4.98% unknown


## 2. Interpolated n-gram

Initialize `InterpolatedNgram` and fit it on the train set. 

In [3]:
ngram_model = InterpolatedNgram(len(vocab))

with Timer() as fit_timer:
    ngram_model.fit(splits["train"])

print(f"Fitted n-gram model in {fit_timer.seconds:.2f} seconds")
print(f"Stored count entries: {ngram_model.stored_count_entries():,}")

# Sparsity: longer histories are more numerous and each is seen less often.
print("\nHistory length | distinct histories | seen once")
for h, totals in enumerate(ngram_model.totals):
    seen_once = sum(1 for count in totals.values() if count == 1)
    print(f"{h:>14} | {len(totals):>18,} | {seen_once:>9,} ({seen_once / len(totals):.1%})")

Fitted n-gram model in 2.02 seconds
Stored count entries: 561,440

History length | distinct histories | seen once
             0 |                  1 |         0 (0.0%)
             1 |              4,102 |         0 (0.0%)
             2 |             56,126 |    36,567 (65.2%)
             3 |            131,276 |   109,829 (83.7%)
             4 |            177,089 |   165,715 (93.6%)


**Add-k smoothing and interpolation: tuning hyperparameters using the validation set**  
Apply grid search to find the best k. For each k value, interpolation weights (1- to 5-gram) are refitted by EM. Negative log-likelihood and perplexity is computed to select the best hyperparameters. 



In [4]:
# Tune add-k (grid) and interpolation weights (EM) on validation
np.set_printoptions(suppress=True)  

k_values = [0.00001, 0.0001, 0.001, 0.01, 0.1, 1.0]
valid_ids = splits["valid"]

best_nll = float("inf") 
best_k = None
best_weights = None
valid_ppls = []  
weights_per_k = []

print("add-k  | valid ppl | weights (1-gram ... 5-gram)")
for k in k_values:
    ngram_model.add_k = k
    ngram_model.fit_weights(valid_ids)
    valid_nll = ngram_model.mean_nll(valid_ids)
    valid_ppl = perplexity(valid_nll)
    valid_ppls.append(valid_ppl)
    weights_per_k.append(ngram_model.weights.copy())
    print(f"{k:<6} | {valid_ppl:>9.2f} | {np.round(ngram_model.weights, 3)}")

    if valid_nll < best_nll:
        best_nll = valid_nll
        best_k = k
        best_weights = ngram_model.weights.copy()

# Use the best settings in the model
ngram_model.add_k = best_k
ngram_model.weights = best_weights
print(f"\nBest add-k = {best_k}, valid perplexity = {perplexity(best_nll):.2f}")
print(f"Weights: {np.round(best_weights, 4)}")

add-k  | valid ppl | weights (1-gram ... 5-gram)
1e-05  |    101.65 | [0.23  0.567 0.152 0.034 0.018]
0.0001 |    101.53 | [0.222 0.571 0.161 0.033 0.012]
0.001  |    103.41 | [0.206 0.608 0.168 0.018 0.001]
0.01   |    114.63 | [0.197 0.714 0.09  0.    0.   ]
0.1    |    146.46 | [0.278 0.722 0.    0.    0.   ]
1.0    |    218.29 | [0.558 0.442 0.    0.    0.   ]

Best add-k = 0.0001, valid perplexity = 101.53
Weights: [0.2224 0.5711 0.1606 0.0335 0.0124]


In [ ]:
weights_array = np.array(weights_per_k)
order_names = ["1-gram", "2-gram", "3-gram", "4-gram", "5-gram"]
order_colors = ["#d62a2a", "#eb6834", "#1baf7a", "#1e63ca", "#e87ba4"] 

fig, (ax_ppl, ax_w) = plt.subplots(1, 2, figsize=(11, 4))

# validation perplexity against add-k
ax_ppl.plot(k_values, valid_ppls, marker="o", color="#2a78d6", linewidth=2)
ax_ppl.scatter([best_k], [perplexity(best_nll)], s=120, facecolors="none",
               edgecolors="black", linewidths=1.5, zorder=3)
ax_ppl.annotate(f"best k = {best_k}\nppl = {perplexity(best_nll):.1f}",
                (best_k, perplexity(best_nll)), textcoords="offset points",
                xytext=(10, 15), fontsize=9)
ax_ppl.set_xscale("log")
ax_ppl.set_xlabel("add-k (log scale)")
ax_ppl.set_ylabel("Validation perplexity")
ax_ppl.set_title("Validation perplexity vs add-k")
ax_ppl.grid(alpha=0.3)

# EM interpolation weight of each order against add-k
for i, (name, color) in enumerate(zip(order_names, order_colors)):
    ax_w.plot(k_values, weights_array[:, i], marker="o", color=color, linewidth=2, label=name)
ax_w.set_xscale("log")
ax_w.set_xlabel("add-k (log scale)")
ax_w.set_ylabel("Interpolation weight (λ)")
ax_w.set_ylim(0, 1)
ax_w.set_title("EM weights vs add-k")
ax_w.legend(frameon=False)
ax_w.grid(alpha=0.3)

fig.tight_layout()
(RESULTS_DIR / "part1").mkdir(parents=True, exist_ok=True)
fig.savefig(RESULTS_DIR / "part1" / "ngram_add_k.png", dpi=200, bbox_inches="tight")
plt.show()

**Test:** Evaluate performance on test set.

In [6]:
test_ids = splits["test"]

with Timer() as test_timer:
    test_nll = ngram_model.mean_nll(test_ids)

test_ppl = perplexity(test_nll)
n_targets = len(test_ids) - 1   # every token except the first is predicted
throughput = n_targets / test_timer.seconds

print(f"add-k = {ngram_model.add_k}, weights = {np.round(ngram_model.weights, 4)}")
print(f"Test NLL: {test_nll:.4f}")
print(f"Test perplexity: {test_ppl:.2f}")
print(f"Scoring time: {test_timer.seconds:.2f} s for {n_targets:,} target tokens, {throughput:,.0f} tokens/s")

add-k = 0.0001, weights = [0.2224 0.5711 0.1606 0.0335 0.0124]
Test NLL: 4.5440
Test perplexity: 94.07
Scoring time: 0.08 s for 9,888 target tokens, 130,955 tokens/s


**Generate Samples:** For each prompt in `PROMPTS`, tokenize and encode the prompt, then sample 30 tokens at temperature 0.8 with the prompt's own seed (4213 + prompt index). Decode back to text.

In [7]:
ngram_samples = generate_samples(ngram_model.next_token_probs, vocab)

for prompt, text in ngram_samples.items():
    print(f"[{prompt}]\n{text}\n")

[i was]
i was born had found them, and france. now!
the time., thinking how long regularly and that the grass, i had not have there

[miss havisham]
miss havisham,, said the time. she was <unk> and stiff, and see nothing., and a <unk> and to say a word pip, and notice

[joe said]
joe said, and the <unk> as he, i felt that you mentioned to the a walk about it rather <unk>, in a so ideas, and that they are

[it was a]
it was a <unk> and you at me, and then we are you about so, making the floor over one of the house, when you to my sister was a



**Save Results:** Save key metrics: test NLL/perplexity, valid NLL, fit time, test time, throughput, stored count entries, chosen add-k and weights, and samples. This writes to `results/part1/ngram.json`.

In [ ]:
ngram_results = {
    "model": "interpolated n-gram (1- to 5-gram, add-k + EM weights)",
    "max_history": ngram_model.max_history,
    "add_k": ngram_model.add_k,
    "weights": ngram_model.weights.tolist(),   
    "add_k_grid": {
        "k_values": k_values,
        "valid_perplexity": valid_ppls,
        "weights": [w.tolist() for w in weights_per_k],
    },
    "valid_nll": best_nll,
    "valid_perplexity": perplexity(best_nll),
    "test_nll": test_nll,
    "test_perplexity": test_ppl,
    "fit_seconds": fit_timer.seconds, 
    "test_seconds": test_timer.seconds,
    "test_tokens_per_second": throughput,
    "stored_count_entries": ngram_model.stored_count_entries(), 
    "samples": ngram_samples,
    "hardware": hardware_summary(),
}

path = save_results("part1/ngram", ngram_results)
print(f"Saved to {path}")

## 3. Neural model settings

Define shared settings for all three neural models. Embedding dimension 100 (to match GloVe 100d, so the RNN/LSTM can be reused in Part II), sequence length 64, batch size 32, Adam with learning rate 1e-3, dropout 0.2, and gradient clipping 1.0.

**Training budget: 40 epochs for every model.** This lets every model reach its best validation loss.

Parameter counts are reported. Most parameters sit in the vocabulary embedding and output layers.

In [9]:
# Shared settings for all three neural models
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 100 # matches GloVe 100d, so the Part I model can be reused in Part II
SEQ_LEN = 64 # tokens per training chunk (also the Transformer's context window)
DROPOUT = 0.2 # same for every model, ~1-2M parameters on ~200k tokens can overfit

TRAINING = dict(
    epochs=40, # 10 and 20 were too few for the LSTM
    seq_len=SEQ_LEN,
    batch_size=32,
    learning_rate=1e-3, # Adam (the Lecture 3 demo used SGD)
    clip=1.0, # gradient-norm clipping
    pad_id=vocab.pad_id,  # padding excluded from the loss
    seed=SEED,
)

# Architecture settings
RECURRENT_CONFIG = dict(embedding_dim=EMBEDDING_DIM, hidden_size=256, num_layers=1,
                        dropout=DROPOUT, pad_id=vocab.pad_id)
TRANSFORMER_CONFIG = dict(embedding_dim=EMBEDDING_DIM, num_heads=4, num_layers=2,
                          feedforward_dim=256, max_length=SEQ_LEN,
                          dropout=DROPOUT, pad_id=vocab.pad_id)
previews = {
    "rnn": RecurrentLM(VOCAB_SIZE, cell="rnn", **RECURRENT_CONFIG),
    "lstm": RecurrentLM(VOCAB_SIZE, cell="lstm", **RECURRENT_CONFIG),
    "transformer": TransformerLM(VOCAB_SIZE, **TRANSFORMER_CONFIG),
}
print("Model sizes (number of parameters)")
print(f"{'model':<12} | {'input embedding':>15} | {'core':>9} | {'output layer':>12} | {'total':>10} | {'core share':>10}")
for name, preview in previews.items():
    total = count_parameters(preview)
    input_embedding = preview.embedding.weight.numel()
    output_layer = sum(p.numel() for p in preview.output.parameters())
    core = total - input_embedding - output_layer
    print(f"{name:<12} | {input_embedding:>15,} | {core:>9,} | {output_layer:>12,} | {total:>10,} | {core / total:>10.1%}")

train_chunks = (len(splits["train"]) - 1) // SEQ_LEN
print(f"\nTraining chunks per epoch: {train_chunks:,} "
      f"({-(-train_chunks // TRAINING['batch_size'])} batches of {TRAINING['batch_size']})")
del previews

Model sizes (number of parameters)
model        | input embedding |      core | output layer |      total | core share
rnn          |         410,300 |    91,648 |    1,054,471 |  1,556,419 |       5.9%
lstm         |         410,300 |   366,592 |    1,054,471 |  1,831,363 |      20.0%
transformer  |         410,300 |   191,312 |      414,403 |  1,016,015 |      18.8%

Training chunks per epoch: 3,111 (98 batches of 32)


## 4. Vanilla RNN

Every neural model (sections 4–6) follows the same two cells:

**1. Build and run:**  
Initialize the model, then `run_neural_model` trains it for 40 epochs, scores the test set, generates samples, and saves key metrics to `results/part1/`.

**2. Loss curve:**  
Plots train and validation loss per epoch with the best epoch marked, and saves the figure to `results/part1/`.

In [ ]:
set_seed(SEED)
rnn_model = RecurrentLM(VOCAB_SIZE, cell="rnn", **RECURRENT_CONFIG)
rnn_results = run_neural_model("part1/rnn", rnn_model, splits, vocab, TRAINING,
                               config=RECURRENT_CONFIG, description="vanilla RNN")

for prompt, text in rnn_results["samples"].items():
    print(f"\n[{prompt}]\n{text}")

In [ ]:
# RNN loss curve
plot_history(rnn_results["history"], "Vanilla RNN: training and validation loss", "part1/rnn_loss.png", label_offset=(15, 40))
plt.show()

## 5. LSTM

Same two cells as the RNN, with `cell="lstm"`. Everything else (settings, training budget, evaluation, decoding) is identical, so differences come from the LSTM layer.

In [ ]:
set_seed(SEED)
lstm_model = RecurrentLM(VOCAB_SIZE, cell="lstm", **RECURRENT_CONFIG)
lstm_results = run_neural_model("part1/lstm", lstm_model, splits, vocab, TRAINING,
                                config=RECURRENT_CONFIG, description="LSTM")

for prompt, text in lstm_results["samples"].items():
    print(f"\n[{prompt}]\n{text}")

In [ ]:
# LSTM loss curve
plot_history(lstm_results["history"], "LSTM: training and validation loss", "part1/lstm_loss.png")
plt.show()

## 6. Causal Transformer

Same training settings, budget, evaluation and decoding as the RNN and LSTM, only the architecture changes (2 attention layers, 4 heads, learned positions, 64-token context). The test set is scored with a sliding window (stride 32), so each prediction sees 32–63 preceding tokens, whereas the RNN/LSTM carry memory through the whole test set.

In [ ]:
set_seed(SEED)
transformer_model = TransformerLM(VOCAB_SIZE, **TRANSFORMER_CONFIG)
transformer_results = run_neural_model("part1/transformer", transformer_model, splits, vocab, TRAINING,
                                       config=TRANSFORMER_CONFIG, description="causal Transformer")

for prompt, text in transformer_results["samples"].items():
    print(f"\n[{prompt}]\n{text}")

In [ ]:
# Transformer loss curve
plot_history(transformer_results["history"], "Causal Transformer: training and validation loss",
             "part1/transformer_loss.png", label_offset=(15, 40))
plt.show()

## 7. Comparison

**Comparison tables:** Load the saved results of all four models. Inference is re-timed for every model under the same conditions (1 warm-up pass, then 5 timed passes over the test set. Its median is reported).  

Prints three tables:  
A. prediction quality, B. training cost and size, C. inference speed.

In [ ]:
MODEL_NAMES = ["ngram", "rnn", "lstm", "transformer"]
LABELS = {"ngram": "n-gram", "rnn": "RNN", "lstm": "LSTM", "transformer": "Transformer"}
saved = {name: json.loads((RESULTS_DIR / "part1" / f"{name}.json").read_text(encoding="utf-8"))
         for name in MODEL_NAMES}

scorers = {
    "ngram": lambda: ngram_model.mean_nll(test_ids),
    "rnn": lambda: neural_mean_nll(rnn_model, test_ids),
    "lstm": lambda: neural_mean_nll(lstm_model, test_ids),
    "transformer": lambda: neural_mean_nll(transformer_model, test_ids),
}
REPEATS = 5
timing = {}
for name, score in scorers.items():
    score() 
    runs = []
    for _ in range(REPEATS):
        with Timer() as t:
            score()
        runs.append(t.seconds)
    timing[name] = {"runs": runs, "median": statistics.median(runs), "min": min(runs), "max": max(runs)}

save_results("part1/inference_timing", {
    "procedure": f"1 warm-up + {REPEATS} timed passes over the test stream, median reported",
    "n_targets": n_targets, "timing": timing, "hardware": hardware_summary(),
})

# Table A: prediction quality
print("A. Prediction quality")
print(f"{'model':<12} | {'test NLL':>8} | {'test ppl':>8} | {'valid ppl':>9} | {'best epoch':>10}")
for name in MODEL_NAMES:
    r = saved[name]
    best_epoch = "-" if name == "ngram" else str(r["best_epoch"])
    print(f"{LABELS[name]:<12} | {r['test_nll']:>8.4f} | {r['test_perplexity']:>8.2f} | "
          f"{r['valid_perplexity']:>9.2f} | {best_epoch:>10}")

# Table B: training cost and size
print("\nB. Training cost and size")
print(f"{'model':<12} | {'total s':>8} | {'to best s':>9} | {'fastest ep s':>12} | {'median ep s':>11} | {'size':>16}")
for name in MODEL_NAMES:
    r = saved[name]
    if name == "ngram":
        print(f"{LABELS[name]:<12} | {r['fit_seconds']:>8.1f} | {'-':>9} | {'-':>12} | {'-':>11} | "
              f"{r['stored_count_entries']:>9,} counts")
    else:
        epochs = r["history"]["epoch_seconds"]
        print(f"{LABELS[name]:<12} | {r['train_seconds']:>8.0f} | {r['seconds_to_best_epoch']:>9.0f} | "
              f"{min(epochs):>12.1f} | {statistics.median(epochs):>11.1f} | {r['trainable_parameters']:>9,} params")

# Table C: inference (re-timed above)
print(f"\nC. Inference: score {n_targets:,} test predictions; median of {REPEATS} passes after 1 warm-up")
print(f"{'model':<12} | {'median s':>8} | {'min-max s':>13} | {'tokens/s':>8}")
for name in MODEL_NAMES:
    t = timing[name]
    print(f"{LABELS[name]:<12} | {t['median']:>8.3f} | {t['min']:>6.3f}-{t['max']:<6.3f} | "
          f"{n_targets / t['median']:>8,.0f}")

In [ ]:
# Loss curves: one panel per neural model
NEURAL = ["rnn", "lstm", "transformer"]
MODEL_COLORS = {"rnn": "#2a78d6", "lstm": "#eb6834", "transformer": "#1baf7a"}   # same colour = same model

fig = plt.figure(figsize=(11, 8))
first = fig.add_subplot(2, 2, 1)
panels = [first, fig.add_subplot(2, 2, 2, sharey=first), fig.add_subplot(2, 2, 3, sharey=first)]
overlay = fig.add_subplot(2, 2, 4)

for ax, name in zip(panels, NEURAL):
    history = saved[name]["history"]
    epochs = range(1, len(history["train_loss"]) + 1)
    best = history["best_epoch"]
    color = MODEL_COLORS[name]
    ax.plot(epochs, history["train_loss"], linestyle="--", color=color, linewidth=2, label="train")
    ax.plot(epochs, history["valid_loss"], linestyle="-", color=color, linewidth=2, label="validation")
    ax.axvline(best, color="grey", linestyle=":", linewidth=1)
    ax.scatter([best], [history["valid_loss"][best - 1]], s=60, facecolors="none",
               edgecolors="black", linewidths=1.2, zorder=3)
    ax.set_title(f"{LABELS[name]} (best epoch {best})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-entropy")
    ax.legend(frameon=False)
    ax.grid(alpha=0.3)

for name in NEURAL:
    history = saved[name]["history"]
    epochs = range(1, len(history["valid_loss"]) + 1)
    best = history["best_epoch"]
    overlay.plot(epochs, history["valid_loss"], color=MODEL_COLORS[name], linewidth=2, label=LABELS[name])
    overlay.scatter([best], [history["valid_loss"][best - 1]], s=60, color=MODEL_COLORS[name],
                    edgecolors="black", linewidths=1.2, zorder=3)
overlay.set_ylim(4.25, 4.85)
overlay.set_title("Validation loss, all three (zoomed; dot = best epoch)")
overlay.set_xlabel("Epoch")
overlay.set_ylabel("Validation cross-entropy")
overlay.legend(frameon=False)
overlay.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "part1" / "loss_curves.png", dpi=200, bbox_inches="tight")
plt.show()

In [18]:
# Generated text: same prompt, temperature 0.8, 30 tokens, seed 4213 + prompt index for every model
for prompt in PROMPTS:
    print("=" * 100)
    print(f"PROMPT: {prompt!r}")
    for name in MODEL_NAMES:
        print(f"\n  [{LABELS[name]}]\n  {saved[name]['samples'][prompt]}")
    print()

PROMPT: 'i was'

  [n-gram]
  i was born had found them, and france. now!
the time., thinking how long regularly and that the grass, i had not have there

  [RNN]
  i was disappointed i must say and i dragged? i cried, and i been <unk>, or over his salt, and i caught the first of the house,

  [LSTM]
  i was succeeded and wondered him.
tar! said estella, and in a <unk> <unk> now.
camilla, said i.
you know a moment,

  [Transformer]
  i was aware of our plan.
j all on this, and i do it, though his lips again, and i wanted, and i have been not

PROMPT: 'miss havisham'

  [n-gram]
  miss havisham,, said the time. she was <unk> and stiff, and see nothing., and a <unk> and to say a word pip, and notice

  [RNN]
  miss havisham, and when i had come into the <unk> and disquiet, and then observed the <unk> <unk>, and <unk> of the clerk and the corduroys <unk>, before

  [LSTM]
  miss havisham, and so.
he has not <unk> you since, he answered at the other <unk>, and, he now the hard of human <un